## NB-05 — The RAG Chain

Retrieval is only half the pipeline. This notebook connects the FAISS index
from NB-04 to Claude using LangChain's `RetrievalQA` chain to build the
generation layer of CatalogueIQ. You will format retrieved chunks with citation
markers, write a system prompt that keeps the model grounded in the knowledge
base, test all five CatalogueIQ query types end-to-end, and implement out-of-
catalogue handling — so the system says "this product is not in our catalogue"
rather than hallucinating specifications.

### Concepts Covered

- Building a `RetrievalQA` chain with `ChatAnthropic` and a FAISS retriever
- Citation format: `[Source: <filename>, Product ID: <id> or Section: <heading>]`
- System prompt design: groundedness constraints for product specs and policy rules
- Handling irrelevant retrievals: detecting when no retrieved chunk answers the question
- Out-of-catalogue product handling: explicit "not found" responses
- Testing all five CatalogueIQ query types with real Claude API responses
- Understanding token costs per query type

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
import pickle
import time
from pathlib import Path
from dotenv import load_dotenv
import faiss
import numpy as np
from langchain.schema import Document
from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS as LangchainFAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
INDEX_DIR = Path("data/index")
os.makedirs("output", exist_ok=True)

# Verify API key is loaded
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found in .env"
print(f"Model: {MODEL}")
print(f"API key: {'✓ loaded' if os.getenv('ANTHROPIC_API_KEY') else '✗ missing'}")


### Step 1 — Load Index and Initialise the LLM

We load the FAISS index saved in NB-04 and wrap the HuggingFace embedder
in a LangChain-compatible class so the retriever can embed queries on the fly.

In [ ]:
# ── LOAD CACHED INDEX ────────────────────────────────────────

index_path = INDEX_DIR / "faiss.index"
meta_path  = INDEX_DIR / "metadata.pkl"

if not index_path.exists():
    raise FileNotFoundError(
        f"{index_path} not found. Run NB-04 first to build the FAISS index."
    )

faiss_index = faiss.read_index(str(index_path))
with open(meta_path, "rb") as f:
    all_docs: list[Document] = pickle.load(f)

print(f"FAISS index loaded: {faiss_index.ntotal} vectors")
print(f"Documents loaded: {len(all_docs)}")

# ── Initialise LangChain components ──────────────────────────

# COST NOTE: ChatAnthropic makes API calls for every generate() call.
llm = ChatAnthropic(
    model=MODEL,
    temperature=0,      # deterministic answers for grounded factual queries
    max_tokens=1024,
)

# Wrap HuggingFace embedder for LangChain compatibility
hf_embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# Build a LangChain FAISS vectorstore from the loaded index + docs
# This gives us .as_retriever() with built-in MMR and metadata filtering
texts = [d.page_content for d in all_docs]
metadatas = [d.metadata for d in all_docs]

vectorstore = LangchainFAISS.from_documents(
    documents=all_docs,
    embedding=hf_embedder,
)
print("LangChain FAISS vectorstore ready.")


### Step 2 — System Prompt with Groundedness Constraints

The system prompt is the most important component for preventing hallucination.
For CatalogueIQ we enforce four hard rules that address the specific failure
modes of an e-commerce assistant:

1. Never invent product specifications (the most common failure)
2. Never invent policy rules — cite the exact section
3. Explicitly acknowledge products not in the catalogue
4. Always include at least one citation per factual claim

In [ ]:
# ── SYSTEM PROMPT ────────────────────────────────────────────

CATALOGUE_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are CatalogueIQ, the product intelligence assistant for
ShopSmart India — a fictional Indian e-commerce marketplace. You help shoppers,
sellers, and support agents get accurate, grounded answers from ShopSmart India's
product catalogue, policy documents, and buyer FAQ.

STRICT RULES — you must follow all of these:
1. NEVER invent or guess product specifications. If a spec is not in the
   retrieved context below, say "this information is not available in the
   current catalogue."
2. NEVER invent policy rules. If a policy is cited, include the exact section
   name from the returns policy or seller guide.
3. If a product is not in the retrieved context, explicitly state:
   "This product does not appear to be in the ShopSmart India catalogue."
   Do not attempt to answer from general knowledge.
4. ALWAYS include at least one citation per factual claim, using this format:
   [Source: <filename>, Product ID: <id> OR Section: <heading>]
5. For comparative queries, compare ALL products found in the retrieved context
   — do not mention only the top result.
6. For policy queries, if the context is ambiguous, flag the ambiguity rather
   than guessing.

RETRIEVED CONTEXT:
{context}

SHOPPER/SELLER QUESTION:
{question}

Answer in clear, helpful English. Use ₹ for Indian Rupee amounts.
Include citations at the end of every sentence that states a fact."""
)

print("System prompt configured with 6 groundedness rules.")
print(f"Prompt template variables: {CATALOGUE_QA_PROMPT.input_variables}")


### Step 3 — Build the RetrievalQA Chain

We build the `RetrievalQA` chain with `top_k=4` (enough to get multiple
products for comparative queries) and `return_source_documents=True` so we
can inspect what context was passed to the model.

In [ ]:
# ── BUILD QA CHAIN ───────────────────────────────────────────

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},     # retrieve 4 chunks per query
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # "stuff" = concatenate all chunks into one context
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": CATALOGUE_QA_PROMPT},
)

print("RetrievalQA chain built.")
print("  LLM: Claude Sonnet 4.6")
print("  Retriever: FAISS similarity, top_k=4")
print("  Chain type: stuff (all chunks concatenated)")

def format_sources(source_docs: list[Document]) -> str:
    """Format retrieved source documents for display."""
    lines = []
    for i, doc in enumerate(source_docs, 1):
        m = doc.metadata
        ref = m.get("product_id") or m.get("section_heading") or m.get("question_number", "")
        key = "Product ID" if m.get("product_id") else "Section"
        lines.append(f"  [{i}] {m.get('source_file', '?')} | {key}: {ref}")
    return "\n".join(lines)


### Step 4 — Test All Five CatalogueIQ Query Types

Each test query triggers a real Claude API call. Watch for:
- Product factual: does the answer cite `products.csv` with a product ID?
- Comparative: does the answer mention multiple products from the context?
- Multi-hop: does the answer integrate chunks from multiple source files?

In [ ]:
# COST NOTE: Each query below makes one API call to Claude.
# 5 queries × ~500 tokens/response = ~$0.05 total at current pricing.

test_queries = [
    ("Product Factual",          "What is the battery life of the boAt Airdopes 141 earbuds?"),
    ("Policy & Eligibility",     "Can I return a fashion item I bought on sale during Big Billion Days?"),
    ("Comparative Recommendation","Which wireless earbuds under ₹3000 have the best noise cancellation?"),
    ("Seller Policy",            "What image resolution do I need for an Electronics listing on ShopSmart India?"),
    ("Multi-Hop",                "I bought a smartwatch from a third-party seller and it stopped working after 45 days. What are my options?"),
]

results_log = []

for query_type, question in test_queries:
    print(f"\n{'='*65}")
    print(f"Query Type: {query_type}")
    print(f"Question: {question}")
    print("─" * 65)

    t0 = time.time()
    # COST NOTE: API call to Claude here
    result = qa_chain.invoke({"query": question})
    elapsed = time.time() - t0

    answer = result["result"]
    sources = result.get("source_documents", [])

    print(f"Answer:\n{answer}")
    print(f"\nSources retrieved ({len(sources)}):")
    print(format_sources(sources))
    print(f"Response time: {elapsed:.1f}s")

    results_log.append({
        "query_type": query_type,
        "question": question,
        "answer": answer,
        "sources": [d.metadata for d in sources],
        "elapsed_s": elapsed,
    })

# Save results
with open("output/nb05_chain_results.json", "w", encoding="utf-8") as f:
    json.dump(results_log, f, indent=2, ensure_ascii=False)
print(f"\nAll results saved to output/nb05_chain_results.json")


### Step 5 — Out-of-Catalogue Product Handling

A critical requirement: when a shopper asks about a product that is not in the
knowledge base, the system must say so explicitly rather than hallucinating specs.
We test this with a product that does not exist in the catalogue.

In [ ]:
# COST NOTE: One API call to Claude for the out-of-catalogue test.

out_of_catalogue_query = (
    "What are the specs of the Apple AirPods Pro (3rd generation) "
    "available on ShopSmart India?"
)

print("Testing out-of-catalogue product handling...")
print(f"Query: {out_of_catalogue_query}\n")

# COST NOTE: API call to Claude
result = qa_chain.invoke({"query": out_of_catalogue_query})
print("Answer:")
print(result["result"])
print("\nSources retrieved:")
print(format_sources(result.get("source_documents", [])))

print("\n✅ Expected behaviour: The model should state that Apple AirPods Pro")
print("   does not appear in the ShopSmart India catalogue.")
print("   If it invents specs — the system prompt needs strengthening.")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Temperature and hallucination risk
# Change the ChatAnthropic temperature to 0.7 and re-run the product factual query.
# Does the model invent any specs that are not in the context?
# Change it back to 0. What does this tell you about temperature choice for
# grounded product assistants?

# EXERCISE 2 — top_k and comparative queries
# Change search_kwargs={"k": 4} to {"k": 1} and re-run the comparative query.
# Does the answer now mention only one product?
# Change to {"k": 8}. Does quality improve, or does noise increase?
# Find the top_k sweet spot for comparative recommendation.

# EXERCISE 3 — Context length vs precision
# The "stuff" chain concatenates all retrieved chunks. For top_k=8 this can
# push the context window. Switch to chain_type="map_reduce" and re-run.
# Compare: does the answer quality change? Is it faster or slower?
# When would you choose map_reduce over stuff for ShopSmart India?

print("Experiments ready. Modify llm temperature, search_kwargs, or chain_type above.")


### Key Takeaway

The RAG chain joins retrieval to generation, but the system prompt is the
critical engineering layer that determines whether Claude stays grounded or
hallucinates. Explicit rules — "never invent specs", "always cite sources",
"acknowledge missing products" — are more effective than vague instructions.
The `return_source_documents=True` flag is essential for debugging: always
check what context the model actually received before blaming the generation layer.

In **NB-06** you will improve retrieval recall through query expansion —
rewriting each shopper query into multiple variants that capture the synonym
diversity of Indian e-commerce language — and add cross-encoder re-ranking
to improve precision on comparative recommendation queries.